# User Churn Classification Case Study

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["PYTHONWARNINGS"] = "ignore"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import statsmodels.formula.api as smf
from scipy import stats

RANDOM_STATE = 42
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

## 1. Load data and perform a minimal global audit


In [ ]:
path_candidates = [
    Path("user_churn_classification_case.csv"),
    Path("/mnt/data/user_churn_classification_case.csv"),
]

path = next((p for p in path_candidates if p.exists()), None)
if path is None:
    raise FileNotFoundError("Expected user_churn_classification_case.csv next to the notebook or under /mnt/data.")

df_raw = pd.read_csv(path, parse_dates=["snapshot_date"])

display(df_raw.head())

print("Using dataset:", path)
print("Raw shape:", df_raw.shape)
print("Duplicate rows:", int(df_raw.duplicated().sum()))
print("Churn rate:", round(df_raw["churn_90d"].mean(), 4))
print("Missing values by column:")
display(df_raw.isna().sum().to_frame("missing_count").T)
print("Dtypes:")
display(df_raw.dtypes.astype(str).to_frame("dtype").T)


In [ ]:
summary_table = pd.DataFrame({
    "min": df_raw.select_dtypes(include=np.number).min(),
    "p01": df_raw.select_dtypes(include=np.number).quantile(0.01),
    "median": df_raw.select_dtypes(include=np.number).median(),
    "p99": df_raw.select_dtypes(include=np.number).quantile(0.99),
    "max": df_raw.select_dtypes(include=np.number).max(),
    "mean": df_raw.select_dtypes(include=np.number).mean(),
    "std": df_raw.select_dtypes(include=np.number).std(),
    "ctn": df_raw.select_dtypes(include=np.number).nunique()
})
display(summary_table)


To be noted: before asking whether a churn model performs well, the first question is whether the target is even well-defined from the dataset. Here the label is named `churn_90d`, which strongly suggests “customer churns within 90 days after the snapshot”, but the table by itself does not tell us whether churn is permanent, whether reactivation is possible, or whether churn means full cancellation versus a softer inactive state.

Also worth noting: every customer appears exactly once (single snapshot date), so this is closer to a one-shot scoring table than a longitudinal panel. We can model `churn_90d`, but we cannot learn about within-customer dynamics or repeated pre-churn trajectories.

In [ ]:
# checking number of dates
df_raw['snapshot_date'].nunique()

--> Date is not a relevant information here, each customer appears only once

To be noted: we could drop the date column

In [ ]:
df = df_raw.drop(columns=["snapshot_date"]).drop_duplicates().copy()
print("Post-cleaning shape:", df.shape)

## 2. Reserve a final holdout before detailed EDA


In [ ]:
dev_df, holdout_df = train_test_split(
    df, test_size=0.20, random_state=RANDOM_STATE, stratify=df["churn_90d"]
)

display(pd.DataFrame({
    "sample": ["development", "final_holdout"],
    "rows": [len(dev_df), len(holdout_df)],
    "churn_rate": [dev_df["churn_90d"].mean(), holdout_df["churn_90d"].mean()],
}))

## 3. EDA on the development sample only


### 3.1 Sanity checks for impossible or suspicious values

In [ ]:
expected_bounds = {
    "tenure_months": (0, np.inf),
    "engagement_score": (0, 100),
    "monthly_logins": (0, np.inf),
    "support_tickets_90d": (0, np.inf),
    "payment_failures_6m": (0, np.inf),
    "price_increase_last_6m": (0, 1),
    "competitor_contact_flag": (0, 1),
    "feature_adoption_score": (0, 100),
    "contract_months_left": (0, np.inf),
    "monthly_arpu": (0, np.inf),
    "churn_90d": (0, 1),
}

sanity_rows = []
for col, (lower, upper) in expected_bounds.items():
    s = dev_df[col]
    invalid = ((s < lower) | (s > upper)) & s.notna()

    sanity_rows.append({
        "column": col,
        "invalid_count": int(invalid.sum()),
        "invalid_pct": round(float(invalid.mean() * 100), 3),
        "observed_min": s.min(skipna=True),
        "observed_max": s.max(skipna=True),
    })

sanity_df = pd.DataFrame(sanity_rows).sort_values(
    ["invalid_count", "invalid_pct"], ascending=False
)
display(sanity_df)

### 3.2 Outlier exploration on numeric features

In [ ]:
numeric_cols = [
    "tenure_months", "engagement_score", "monthly_logins", "support_tickets_90d",
    "payment_failures_6m", "feature_adoption_score", "contract_months_left", "monthly_arpu"
]

outlier_rows = []
for col in numeric_cols:
    s = dev_df[col]
    s_non_null = s.dropna()

    if s_non_null.std() > 0:
        z_scores = (s_non_null - s_non_null.mean()) / s_non_null.std()
        outlier_pct = (z_scores.abs() > 3).mean() * 100
    else:
        outlier_pct = 0.0

    outlier_rows.append({
        "feature": col,
        "p01": s_non_null.quantile(0.01),
        "p99": s_non_null.quantile(0.99),
        "mean": s_non_null.mean(),
        "std": s_non_null.std(),
        "abs_z_gt_3_pct": round(float(outlier_pct), 2),
    })

outlier_df = pd.DataFrame(outlier_rows).sort_values("abs_z_gt_3_pct", ascending=False)
display(outlier_df)

In [ ]:
%matplotlib inline

fig, axes = plt.subplots(len(numeric_cols), 2, figsize=(12, 3 * len(numeric_cols)))
for i, col in enumerate(numeric_cols):
    axes[i, 0].boxplot(dev_df[col].dropna(), vert=False)
    axes[i, 0].set_title(f"Boxplot - {col}")
    axes[i, 1].hist(dev_df[col].dropna(), bins=30)
    axes[i, 1].set_title(f"Histogram - {col}")
plt.tight_layout()
plt.show()

The outlier check looks broadly reassuring. Extreme values are present, but their frequency remains limited and the affected variables are precisely those where right tails are economically plausible (support tickets, payment failures, tenure).

I therefore do not see a strong case for removing observations at this stage. This looks more like natural skewness than a genuine data-quality issue.

### 3.3 Missing values: volume and selectivity

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": dev_df.isna().sum(),
    "missing_pct": dev_df.isna().mean().mul(100).round(2),
}).sort_values(["missing_count", "missing_pct"], ascending=False)
display(missing_summary)

missing_by_target = {}
for col in dev_df.columns:
    missing_by_target[col] = dev_df.groupby("churn_90d")[col].apply(lambda s: s.isna().mean())
missing_by_target = pd.DataFrame(missing_by_target).T
missing_by_target.columns = ["missing_rate_non_churn", "missing_rate_churn"]
missing_by_target["gap_churn_minus_non_churn"] = missing_by_target["missing_rate_churn"] - missing_by_target["missing_rate_non_churn"]
display(missing_by_target.sort_values("gap_churn_minus_non_churn", ascending=False).head(15))


--> Missingness differs materially by target: churners have roughly 2-3x higher missing rates on engagement, adoption and support ticket variables. This suggests that the absence of a value may itself be a risk signal, which motivates creating explicit missingness indicator features later in the feature engineering step.

### 3.4 Target balance and class imbalance

In [ ]:
print(f"Churn rate: {dev_df['churn_90d'].mean():.4f}")
print(f"Churners: {int(dev_df['churn_90d'].sum())} / {len(dev_df)}")

fig, ax = plt.subplots(figsize=(5, 3))
dev_df["churn_90d"].value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_xticklabels(["Non-churn", "Churn"], rotation=0)
ax.set_title("Target distribution")
plt.tight_layout()
plt.show()

Class imbalance is extreme: ~0.4% churn rate, meaning roughly 38 churners in the development sample. This has several direct consequences for the analysis:

- Raw accuracy is meaningless (a trivial "predict no churn" achieves 99.6%)
- Any holdout evaluation will be noisy due to very few positive events
- Model selection should rely on ranking metrics (average precision, AUC) rather than threshold-dependent ones
- Class rebalancing will be necessary during training

### 3.5 Churn rate by categorical segment

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

dev_df.groupby("plan_type")["churn_90d"].mean().sort_values().plot(kind="bar", ax=axes[0])
axes[0].set_title("Churn rate by plan")

dev_df.groupby("acquisition_channel")["churn_90d"].mean().sort_values().plot(kind="bar", ax=axes[1])
axes[1].set_title("Churn rate by channel")

dev_df.groupby("region")["churn_90d"].mean().sort_values().plot(kind="bar", ax=axes[2])
axes[2].set_title("Churn rate by region")

plt.tight_layout()
plt.show()

# Cross-tabulation: plan x channel
segment_rate = (
    dev_df.groupby(["plan_type", "acquisition_channel"])["churn_90d"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)
display(segment_rate)

# Region x channel
region_channel = (
    dev_df.groupby(["region", "acquisition_channel"])["churn_90d"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)
display(region_channel.head(10))

The categorical breakdown shows a clear gradient: premium customers churn less than basic, and paid acquisition is materially riskier than organic or partner channels. Region has a weaker effect and looks more like a secondary segmentation variable.

The cross-tabulation reveals that churn concentrates in specific plan x channel combinations (e.g. basic + paid). This concentration could reflect acquisition-quality differences rather than ongoing customer health — a point worth investigating further with the business team.

Given the relatively modest number of categories and the clear monotonic patterns, I do not expect complex categorical interactions to add much beyond what logistic regression can capture natively through one-hot encoding.

### 3.6 Churners vs non-churners: univariate profiling

In [ ]:
profile_rows = []
for col in numeric_cols:
    g0 = dev_df.loc[dev_df["churn_90d"] == 0, col].dropna()
    g1 = dev_df.loc[dev_df["churn_90d"] == 1, col].dropna()

    profile_rows.append({
        "feature": col,
        "mean_non_churn": g0.mean(),
        "mean_churn": g1.mean(),
        "mean_diff": g1.mean() - g0.mean(),
    })

profile_df = pd.DataFrame(profile_rows).sort_values("mean_diff", ascending=False)
display(profile_df)

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 16))
for ax, col in zip(axes.flatten(), numeric_cols):
    sub0 = dev_df.loc[dev_df["churn_90d"] == 0, col].dropna()
    sub1 = dev_df.loc[dev_df["churn_90d"] == 1, col].dropna()
    ax.hist(sub0.values, bins=30, density=True, alpha=0.6, label="non-churn")
    ax.hist(sub1.values, bins=30, density=True, alpha=0.6, label="churn")
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()


The churner profile looks economically sensible. Churners are, on average, less engaged, less adopted, shorter-tenure customers with fewer contract months left, while also showing more support friction and more payment failures.

At this stage, I view this as a useful descriptive check: the model is being fed variables that move in the expected direction. I would not, however, interpret the raw mean differences as a formal ranking of feature importance.

### 3.7 Pairwise relationships, redundancy, and interaction candidates

In [ ]:
corr_cols = [
    "tenure_months", "engagement_score", "monthly_logins", "support_tickets_90d",
    "payment_failures_6m", "feature_adoption_score", "contract_months_left",
    "monthly_arpu", "churn_90d"
]
plt.figure(figsize=(9, 7))
sns.heatmap(dev_df[corr_cols].corr(numeric_only=True), annot=True, cmap="coolwarm", center=0)
plt.title("Numeric correlation screen")
plt.tight_layout()
plt.show()


In [ ]:
# Inspect specific interaction candidates
# 1. Friction: support tickets + payment failures combined
dev_df["_friction"] = (
    (dev_df["support_tickets_90d"].fillna(0) / 3.0) + 
    (dev_df["payment_failures_6m"] / 2.0)
)

# 2. Low engagement x short contract
dev_df["_low_engage_short_contract"] = (
    (dev_df["engagement_score"].fillna(50) < 45).astype(int) *
    (dev_df["contract_months_left"] < 6).astype(int)
)

# 3. Low adoption x competitor contact
dev_df["_low_adopt_competitor"] = (
    (dev_df["feature_adoption_score"].fillna(50) < 50).astype(int) *
    (dev_df["competitor_contact_flag"] == 1).astype(int)
)

interaction_candidates = {
    "friction_score": "_friction",
    "low_engage_short_contract": "_low_engage_short_contract",
    "low_adopt_competitor": "_low_adopt_competitor",
}

int_rows = []
for name, col in interaction_candidates.items():
    if dev_df[col].nunique() <= 2:
        g0 = dev_df.loc[dev_df[col] == 0, "churn_90d"].mean()
        g1 = dev_df.loc[dev_df[col] == 1, "churn_90d"].mean()
        n1 = int((dev_df[col] == 1).sum())
        int_rows.append({"interaction": name, "churn_rate_0": g0, "churn_rate_1": g1, "n_flagged": n1, "lift": g1 / g0 if g0 > 0 else np.nan})
    else:
        corr = dev_df[col].corr(dev_df["churn_90d"])
        int_rows.append({"interaction": name, "churn_rate_0": np.nan, "churn_rate_1": np.nan, "n_flagged": np.nan, "lift": np.nan, "corr_with_churn": corr})

display(pd.DataFrame(int_rows))

# clean up temp columns
dev_df.drop(columns=["_friction", "_low_engage_short_contract", "_low_adopt_competitor"], inplace=True)

There is visible redundancy between engagement, logins and feature adoption — they likely proxy the same latent "customer health" concept. Coefficient interpretation should remain cautious.

The interaction analysis is more interesting. low_engage_short_contract shows a 7.3x lift over the base churn rate on 196 customers — the strongest univariate signal in the entire EDA. low_adopt_competitor also shows a 5.1x lift but on a smaller subgroup (n=104). friction_score has a modest 0.045 correlation with churn, consistent with the earlier univariate findings.

I will create low_engage_short_contract and friction_score as features, along with the missingness indicators from section 3.3. low_adopt_competitor is left out for parsimony but worth revisiting.

## 4. Feature engineering and significance

This section engineers new features motivated by EDA findings, defines the feature set, and runs a logistic regression with robust standard errors to assess statistical significance before moving to ML models.

Before creating new features, I would explicitly verify in a real setting that each existing field is known at snapshot time and not generated after churn risk materializes. For instance, a "retention_offer_accepted" variable or a post-cancellation support interaction would be obvious leakage. I do not see red flags in the current dataset but flag this as a due-diligence step.

### 4.1 Engineered features

In [ ]:
def add_features(data):
    out = data.copy()
    # Missingness indicators (motivated by §3.3: missingness correlated with churn)
    for col in ["engagement_score", "feature_adoption_score", "support_tickets_90d"]:
        out[f"{col}_missing"] = out[col].isna().astype(int)
    # Friction score (motivated by §3.7: support tickets + payment failures combined)
    out["friction_score"] = (
        (out["support_tickets_90d"].fillna(0) / 3.0) +
        (out["payment_failures_6m"] / 2.0)
    )
    # Low engagement × short contract (motivated by §3.7: 7.3× lift)
    out["low_engage_short_contract"] = (
        (out["engagement_score"].fillna(50) < 45).astype(int) *
        (out["contract_months_left"] < 6).astype(int)
    )
    return out

dev_fe = add_features(dev_df)
holdout_fe = add_features(holdout_df)

print("New features: missingness indicators (3), friction_score, low_engage_short_contract")
print(f"Churn rate where low_engage_short_contract == 1: "
      f"{dev_fe.loc[dev_fe['low_engage_short_contract'] == 1, 'churn_90d'].mean():.4f}")
print(f"Churn rate where low_engage_short_contract == 0: "
      f"{dev_fe.loc[dev_fe['low_engage_short_contract'] == 0, 'churn_90d'].mean():.4f}")

In [ ]:
target = "churn_90d"
excluded_cols = {"customer_id", "churn_90d"}
features = [c for c in dev_fe.columns if c not in excluded_cols]

numeric_features = dev_fe[features].select_dtypes(include=np.number).columns.tolist()
categorical_features = dev_fe[features].select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Features: {len(features)} total — {len(numeric_features)} numeric, {len(categorical_features)} categorical")

### 4.2 Logistic regression with robust standard errors

In [ ]:
sig_df = dev_fe.copy()

# Impute for statsmodels (no built-in NA handling)
for col in categorical_features:
    sig_df[col] = sig_df[col].fillna("Missing")
for col in numeric_features:
    if sig_df[col].isna().any():
        sig_df[col] = sig_df[col].fillna(sig_df[col].median())

logit_formula = target + " ~ " + " + ".join(
    numeric_features + [f"C({c})" for c in categorical_features]
)

logit_sig = smf.logit(formula=logit_formula, data=sig_df).fit(cov_type="HC1", disp=False)
print(logit_sig.summary())

The logistic regression (pseudo-R² = 0.19, 9,600 observations, 22 parameters) confirms the directional findings from EDA but also exposes the limits of formal inference with so few events.

**Quasi-complete separation.** Statsmodels flags that 11% of observations can be perfectly predicted, producing `NaN` standard errors for four features: `support_tickets_90d`, `payment_failures_6m`, `support_tickets_90d_missing`, and `friction_score`. These features, or combinations of them, perfectly separate a subset of non-churners, causing the MLE to diverge. Their coefficients are not identifiable in this specification.

**Significant features** (among those with estimable SEs): `tenure_months` (−0.063, p = 0.001), `competitor_contact_flag` (+1.39, p < 0.001), `price_increase_last_6m` (+1.02, p = 0.005), `engagement_score` (−0.044, p = 0.044), and `engagement_score_missing` (+1.56, p = 0.037). These are the most credible effects — directionally consistent with EDA and robust to the low event count.

**Non-significant features**: `plan_type` (p = 0.74/0.85), all `region` dummies (p > 0.7), all `acquisition_channel` dummies (p > 0.2), `feature_adoption_score` (p = 0.26), `contract_months_left` (p = 0.22), `monthly_arpu` (p = 0.56), `low_engage_short_contract` (p = 0.82). With EPV ≈ 1.7, these non-significances reflect insufficient power, not necessarily absent effects.

**No feature pruning.** Unlike the temporal churn notebook where EPV ≈ 60 allowed confident pruning of 4 redundant features, the significance test here lacks the statistical power to distinguish signal from noise. Several non-significant features in the logit carry substantial importance in the random forest (`feature_adoption_score`: 14%, `contract_months_left`: 10.7%, `friction_score`: 10.8%). Pruning on fragile p-values would risk removing genuinely useful predictors. The 18-feature set is already lean, and L2 regularization handles redundancy during model training.

Unlike the temporal churn notebook, clustering is not needed here — each customer appears exactly once in the dataset, so observations are independent.

## 5. Preprocessing and candidate models

In [ ]:
# ── Train/test split (75/25 of dev, stratified) ──
train_df, test_df = train_test_split(
    dev_fe, test_size=0.25, random_state=7, stratify=dev_fe[target]
)

X_train = train_df[features].copy()
y_train = train_df[target].copy()
X_test = test_df[features].copy()
y_test = test_df[target].copy()
X_holdout = holdout_fe[features].copy()
y_holdout = holdout_fe[target].copy()

# ── Preprocessing pipelines ──
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)

# Sample weights for GB (no native class_weight)
churn_ratio = y_train.mean()
sw = np.where(y_train == 1, (1 - churn_ratio) / churn_ratio, 1.0)

print("Train:", len(X_train), "| Test:", len(X_test), "| Holdout:", len(X_holdout))
print("Features:", len(features))

**Cross-validation strategy: `StratifiedKFold`.** With an extreme class imbalance (~0.4% churn, ~38 events in dev), stratification is essential to ensure each fold contains at least some positive events — without it, some folds would have zero churners and AP would be undefined. Unlike the temporal churn notebook where `TimeSeriesSplit` was needed to prevent temporal leakage, this dataset is a single cross-sectional snapshot with no temporal dimension, so stratified random splitting is appropriate.

In [ ]:
model_grids = {
    "logistic_regression": {
        "model": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "param_grid": {
            "model__C": [0.1, 1.0, 5.0],
            "model__penalty": ["l1", "l2"],
            "model__solver": ["liblinear"],
        },
        "fit_params": {},
    },

    "random_forest": {
        "model": RandomForestClassifier(random_state=42, class_weight="balanced"),
        "param_grid": {
            "model__n_estimators": [200, 400],
            "model__max_depth": [4, 8, None],
            "model__min_samples_leaf": [5, 20],
            "model__max_features": ["sqrt", 0.5],
        },
        "fit_params": {},
    },

    "gradient_boosting": {
        "model": GradientBoostingClassifier(random_state=42),
        "param_grid": {
            "model__n_estimators": [100, 200],
            "model__learning_rate": [0.05, 0.1],
            "model__max_depth": [2, 3],
            "model__subsample": [0.8, 1.0],
        },
        # GB does not support class_weight; we pass sample_weight instead
        "fit_params": {"model__sample_weight": sw},
    },
}

search_results = []
best_models = {}

for model_name, spec in model_grids.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", spec["model"])
    ])

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=spec["param_grid"],
        scoring="average_precision",
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=0,
    )

    grid.fit(X_train, y_train, **spec["fit_params"])

    best_model = grid.best_estimator_
    test_proba = best_model.predict_proba(X_test)[:, 1]

    search_results.append({
        "model": model_name,
        "best_cv_avg_precision": grid.best_score_,
        "test_avg_precision": average_precision_score(y_test, test_proba),
        "test_roc_auc": roc_auc_score(y_test, test_proba),
        "best_params": grid.best_params_,
    })

    best_models[model_name] = best_model

results_df = pd.DataFrame(search_results).sort_values(
    "test_avg_precision", ascending=False
).reset_index(drop=True)

display(results_df)

Logistic regression edges out with the best test AP (0.012) and AUC (0.78), followed by RF (AP 0.010, AUC 0.70) and GB (AP 0.007, AUC 0.66). The best logistic config uses L2 at C=1.0 — moderate regularization, less aggressive than the L1/C=0.1 selected in the temporal churn notebook, consistent with fewer features and more collinearity.

The absolute AP values are very low, but this is expected with a 0.4% base rate — AP is bounded below by the base rate, and the relevant comparison is whether the model beats random (AP ≈ 0.004). All three models do.

For churn, I would usually privilege the simpler model unless the nonlinear one brings a clear lift. Logistic regression is easier to explain to an operations team and often surprisingly competitive on tabular churn data.

Note on Gradient Boosting: unlike logistic regression and random forest, `GradientBoostingClassifier` does not support `class_weight` natively. I therefore pass explicit `sample_weight` to re-weight the minority class during training. Without this, the model would largely ignore the rare churn events.

## 6. Decile analysis for an actioning strategy

### 6a. Decile analysis

In [ ]:
logistic_test_proba = best_models["logistic_regression"].predict_proba(test_df[features])[:, 1]

tmp = test_df[["churn_90d"]].copy()
tmp["score"] = logistic_test_proba
tmp["score_rank_pct"] = tmp["score"].rank(pct=True)

base_rate = tmp["churn_90d"].mean()

bucket_rows = []
for lower in range(0, 100, 10):
    upper = lower + 10

    pct_low = 1 - upper / 100
    pct_high = 1 - lower / 100

    if lower == 0:
        sub = tmp[(tmp["score_rank_pct"] >= pct_low) & (tmp["score_rank_pct"] <= pct_high)]
    else:
        sub = tmp[(tmp["score_rank_pct"] >= pct_low) & (tmp["score_rank_pct"] < pct_high)]

    churn_rate = sub["churn_90d"].mean() if len(sub) > 0 else np.nan

    bucket_rows.append({
        "bucket": f"{lower}-{upper}%",
        "n_obs": len(sub),
        "n_churn": int(sub["churn_90d"].sum()),
        "observed_churn_rate": churn_rate,
        "lift_vs_base": churn_rate / base_rate if base_rate > 0 and pd.notna(churn_rate) else np.nan,
    })

bucket_df = pd.DataFrame(bucket_rows)
display(bucket_df)

The decile analysis shows noisy but directionally correct ranking. The top two deciles capture 5 of 9 test churners (lift 2.2× and 3.3×), while the bottom 5 deciles contain zero churners. The ranking is not monotone (decile 2 outperforms decile 1), which is expected with only 9 positive events.

The model appears more useful as a coarse prioritization tool — flagging the top 20% to capture the majority of churners — than as a precise individual-level prediction.

### 6b. Precision-Recall curve and operational threshold sweep

Since the base rate is ~0.4%, a confusion matrix at a fixed threshold is not very informative. A more useful view is: if we contact the top N% of customers by score, how many actual churners do we capture, and what is the precision of that outreach?

In [ ]:
from sklearn.metrics import precision_recall_curve

logistic_test_proba = best_models["logistic_regression"].predict_proba(test_df[features])[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, logistic_test_proba)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PR curve
axes[0].plot(recall, precision)
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall curve (logistic regression, test set)")
axes[0].axhline(y_test.mean(), linestyle="--", color="gray", label=f"base rate = {y_test.mean():.4f}")
axes[0].legend()

# Operational sweep: if we contact top k%, what do we get?
sweep_rows = []
n_total = len(test_df)
n_churn = int(y_test.sum())

for top_pct in [1, 2, 3, 5, 10, 15, 20, 30, 50]:
    k = max(1, int(n_total * top_pct / 100))
    top_idx = np.argsort(-logistic_test_proba)[:k]
    captured = int(y_test.iloc[top_idx].sum())
    sweep_rows.append({
        "contact_top_pct": f"{top_pct}%",
        "n_contacted": k,
        "churners_captured": captured,
        "recall": captured / n_churn if n_churn > 0 else 0,
        "precision": captured / k,
    })

sweep_df = pd.DataFrame(sweep_rows)
display(sweep_df)

axes[1].plot(sweep_df["n_contacted"], sweep_df["recall"], marker="o")
axes[1].set_xlabel("Number of customers contacted")
axes[1].set_ylabel("Recall (fraction of churners captured)")
axes[1].set_title("Operational trade-off: outreach volume vs churn capture")
plt.tight_layout()
plt.show()

The operational sweep confirms the prioritization framing: contacting the top 5% captures 22% of churners, the top 15% captures 44%, and the top 20% captures 56%. Precision remains low in absolute terms (1–2%), which is mechanical given the 0.4% base rate, but the recall curve shows that the model concentrates risk effectively in the upper score range.

In practice, the choice of threshold depends on the cost of a retention intervention versus the expected revenue loss from a churner. Without that cost structure, a top-15–20% outreach appears reasonable as a starting point.

## 7. Inspect the models

In [ ]:
logit_feature_names = best_models["logistic_regression"].named_steps["prep"].get_feature_names_out()
logit_coefs = pd.Series(
    best_models["logistic_regression"].named_steps["model"].coef_[0],
    index=logit_feature_names
).sort_values(key=np.abs, ascending=False)

display(logit_coefs.head(20).to_frame("logit_coef"))


rf_feature_names = best_models["random_forest"].named_steps["prep"].get_feature_names_out()
rf_imp = pd.Series(
    best_models["random_forest"].named_steps["model"].feature_importances_,
    index=rf_feature_names
).sort_values(ascending=False)

display(rf_imp.head(20).to_frame("rf_importance"))

The coefficient structure is economically sensible. On the logistic side, `plan_type_premium` (−1.19) and `tenure_months` (−1.19) are the strongest protective effects, followed by `plan_type_standard` (−1.11) and `engagement_score` (−0.67). On the risk side, `sales` acquisition (+0.71), `support_tickets_90d_missing` (+0.64), `competitor_contact_flag` (+0.62) and `paid` acquisition (+0.59) increase predicted churn.

The random forest importance ranking tells a complementary story: `engagement_score` (21.4%), `feature_adoption_score` (14.0%), `tenure_months` (13.2%), `friction_score` (10.8%), and `contract_months_left` (10.7%) dominate. Notably, `friction_score` ranks 4th in the RF despite having `NaN` SEs in the logit — illustrating how structural redundancy with its components doesn't prevent it from being useful as a composite split in trees.

Both models agree on the core drivers: engagement, tenure, and friction. The categoricals matter more in the logistic (plan, channel), while the RF relies more on continuous behavioral variables.

## 8. Final holdout evaluation and significance

### 8a. Holdout evaluation

In [ ]:
hold_rows = []
for name, model in best_models.items():
    proba = model.predict_proba(holdout_fe[features])[:, 1]

    hold_rows.append({
        "model": name,
        "avg_precision": average_precision_score(holdout_fe["churn_90d"], proba),
        "roc_auc": roc_auc_score(holdout_fe["churn_90d"], proba),
    })

holdout_results_df = pd.DataFrame(hold_rows).sort_values(
    "avg_precision", ascending=False
).reset_index(drop=True)

display(holdout_results_df)

In [ ]:
logistic_holdout_proba = best_models["logistic_regression"].predict_proba(holdout_fe[features])[:, 1]

tmp = holdout_fe[["churn_90d"]].copy()
tmp["score"] = logistic_holdout_proba
tmp["score_rank_pct"] = tmp["score"].rank(pct=True)

base_rate = tmp["churn_90d"].mean()

bucket_rows = []
for lower in range(0, 100, 10):
    upper = lower + 10

    pct_low = 1 - upper / 100
    pct_high = 1 - lower / 100

    if lower == 0:
        sub = tmp[(tmp["score_rank_pct"] >= pct_low) & (tmp["score_rank_pct"] <= pct_high)]
    else:
        sub = tmp[(tmp["score_rank_pct"] >= pct_low) & (tmp["score_rank_pct"] < pct_high)]

    churn_rate = sub["churn_90d"].mean() if len(sub) > 0 else np.nan

    bucket_rows.append({
        "bucket": f"{lower}-{upper}%",
        "n_obs": len(sub),
        "n_churn": int(sub["churn_90d"].sum()),
        "observed_churn_rate": churn_rate,
        "lift_vs_base": churn_rate / base_rate if base_rate > 0 and pd.notna(churn_rate) else np.nan,
    })

holdout_bucket_df = pd.DataFrame(bucket_rows)
display(holdout_bucket_df)

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(holdout_bucket_df["bucket"], holdout_bucket_df["lift_vs_base"])
plt.axhline(1.0, linestyle="--", label="base rate")
plt.xticks(rotation=45)
plt.ylabel("Lift vs base")
plt.title("Holdout lift by score decile (logistic_regression)")
plt.legend()
plt.tight_layout()
plt.show()

On holdout, logistic regression remains the strongest model: AP = 0.056, AUC = 0.81. RF follows at AP = 0.012, AUC = 0.78, and GB at AP = 0.009, AUC = 0.66.

The holdout AP (0.056) is notably higher than the test AP (0.012). This is not a sign of improvement — with only 9 churners in each set, a single positive landing in the right decile swings AP substantially. The bootstrap CI [0.007, 0.269] confirms that both estimates fall within the same uncertainty band.

The holdout lift chart shows the model has ranking value: the top decile captures 4 of 9 churners (lift 4.4×), and deciles 2–3 capture another 4. From decile 4 onward, nearly all buckets contain zero churners. The score is relevant for top-of-book prioritization but not for fine-grained ranking across the full customer base.

### 8b. Bootstrap uncertainty on holdout metrics

With only ~9 churn events in the holdout, point estimates of average precision and lift are statistically fragile. A single churner moving between deciles can shift the top-decile lift by 20% or more.

To quantify this uncertainty, I use a nonparametric bootstrap: I resample the holdout with replacement 2,000 times, recompute the metric on each resample, and build an empirical confidence interval from the resulting distribution. This approach makes no distributional assumption about the metrics (unlike a t-test, which assumes normality) and naturally accounts for the extreme class imbalance — resamples where few or no churners are drawn will produce low AP values, while resamples that happen to concentrate churners in the top decile will produce high lift.

The width of the resulting confidence interval directly answers the question: "how much should I trust the holdout numbers?".

In [ ]:
np.random.seed(42)
n_boot = 2000

# predict_proba returns [P(class=0), P(class=1)], so we take column 1.
logistic_holdout_proba = best_models["logistic_regression"].predict_proba(holdout_fe[features])[:, 1]

# Extract true labels from the holdout set as a NumPy array.
y_hold = holdout_fe["churn_90d"].values

# Will store Average Precision values across bootstrap samples.
boot_ap = []

# Will store top-decile lift values across bootstrap samples.
boot_lift_top10 = []

for _ in range(n_boot):

    # Sample indices with replacement (same size as original data).
    # Some observations may appear multiple times, others not at all.
    idx = np.random.choice(len(y_hold), size=len(y_hold), replace=True)

    # Bootstrap sample of true labels.
    y_b = y_hold[idx]

    # Bootstrap sample of predicted probabilities.
    p_b = logistic_holdout_proba[idx]
    
    # Skip samples with no positive class (AP is undefined in that case).
    if y_b.sum() == 0:
        continue
    
    # Compute Average Precision for this bootstrap sample.
    boot_ap.append(average_precision_score(y_b, p_b))
    
    # Number of observations in the top decile (top 10% by predicted probability).
    # Ensure at least one observation is selected.
    k = max(1, len(y_b) // 10)

    # Get indices of the k highest predicted probabilities.
    top_idx = np.argsort(-p_b)[:k]

    # Observed positive rate in the top decile.
    top_rate = y_b[top_idx].mean()

    # Overall positive rate in the bootstrap sample.
    base_rate = y_b.mean()

    # Top-decile lift: ratio of top rate to base rate.
    # If base_rate is zero, return NaN to avoid division by zero.
    boot_lift_top10.append(top_rate / base_rate if base_rate > 0 else np.nan)

# Convert lists to NumPy arrays for easier numerical operations.
boot_ap = np.array(boot_ap)

# Remove NaN values from lift before converting to array.
boot_lift = np.array([x for x in boot_lift_top10 if not np.isnan(x)])

print("Holdout Average Precision")
print(f"  Point estimate: {average_precision_score(y_hold, logistic_holdout_proba):.4f}")
print(f"  Bootstrap 95% CI: [{np.percentile(boot_ap, 2.5):.4f}, {np.percentile(boot_ap, 97.5):.4f}]")
print()
print("Top-decile lift")
print(f"  Point estimate: {holdout_bucket_df.iloc[0]['lift_vs_base']:.2f}x")
print(f"  Bootstrap 95% CI: [{np.percentile(boot_lift, 2.5):.2f}x, {np.percentile(boot_lift, 97.5):.2f}x]")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(boot_ap, bins=40, edgecolor="white")
axes[0].axvline(np.percentile(boot_ap, 2.5), color="red", linestyle="--", label="2.5%")
axes[0].axvline(np.percentile(boot_ap, 97.5), color="red", linestyle="--", label="97.5%")
axes[0].set_title("Bootstrap distribution: Average Precision")
axes[0].legend()

axes[1].hist(boot_lift, bins=40, edgecolor="white")
axes[1].axvline(np.percentile(boot_lift, 2.5), color="red", linestyle="--", label="2.5%")
axes[1].axvline(np.percentile(boot_lift, 97.5), color="red", linestyle="--", label="97.5%")
axes[1].set_title("Bootstrap distribution: Top-decile lift")
axes[1].legend()
plt.tight_layout()
plt.show()

The bootstrap confirms real but imprecise signal. The top-decile lift CI [1.1×, 8.0×] stays above 1× in nearly all resamples — the ranking value is not an artifact. But the AP CI [0.007, 0.269] spans a 40× range, meaning 9 holdout events are simply not enough to pin down performance precisely.

The next step is to monitor rolling performance as new churn data accumulates, building statistical confidence over time rather than relying on a single holdout evaluation.

## Final remarks

The main finding is that a simple logistic regression, trained with class rebalancing on a modest set of behavioral and contractual features, produces a score with genuine ranking value for churn prediction. The top decile is consistently enriched across both development and holdout samples, and the model's coefficient structure aligns with economically plausible risk drivers: short tenure, low engagement, high friction, and competitor contact all increase predicted churn risk.

However, several important caveats apply:

**Statistical fragility.** With a base rate of ~0.4%, the holdout contains only ~9 churn events. Bootstrap confidence intervals on the key metrics are wide, and point estimates should not be over-interpreted. The model is better understood as directionally correct than as precisely calibrated.

**The score is a prioritization tool, not a prediction model.** Given the extreme class imbalance, the most realistic operational use is to flag the top 5-10% of customers for proactive retention outreach, not to predict individual churn probabilities. The PR curve and threshold sweep support this framing.

**What I would do next:**
- Monitor the score's lift on a rolling basis as new churn events accumulate, to build statistical confidence over time
- Test calibration more formally (Platt scaling or isotonic regression) if the score is to be used for expected-value calculations rather than pure ranking
- Investigate whether a longitudinal version of the dataset (multiple snapshots per customer) would allow trajectory-based features, which tend to be among the strongest churn predictors in practice
- Conduct a cost-benefit analysis with the operations team to set an explicit contact threshold